In [55]:
import pandas as pd
import numpy as np
from scipy.stats import chisquare
import re
# Librerías gráficas
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocesamiento y Modelado
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder,MinMaxScaler
from sklearn.feature_selection import mutual_info_classif
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

# Configuración global
np.random.seed(1989)
pd.set_option("display.float_format", '{:,.2f}'.format)

print("Bibliotecas importadas correctamente.")

Bibliotecas importadas correctamente.


In [13]:
class analyze:
    # Inicializamos la clase
    def __init__(self, df):
        self.df = df

    # Método para obtener las estadísticas detalladas
    def stats(self, x):
        # Filtramos nulos para cálculos estadísticos
        series = self.df[x].dropna()
        if series.empty:
            return pd.DataFrame()

        minimo = np.round(np.min(series), 4)
        p01 = np.round(np.percentile(series, 1), 4)
        p05 = np.round(np.percentile(series, 5), 4)
        media = np.round(np.mean(series), 4)
        mediana = np.round(np.median(series), 4)
        p95 = np.round(np.percentile(series, 95), 4)
        p99 = np.round(np.percentile(series, 99), 4)
        maximo = np.round(np.max(series), 4)
        varianza = np.round(np.var(series, ddof=1), 4)
        ds = np.round(np.std(series, ddof=1), 4)
        kurt = series.kurtosis()
        skew = series.skew()

        data = {
            'Estadísticas': ["minimo", "P01%", "P05%", "media", "mediana", "P95%", "P99%", "maximo", "varianza", "desv_std", "kurtosis", "skewness"],
            'Valores': [minimo, p01, p05, media, mediana, p95, p99, maximo, varianza, ds, kurt, skew]
        }
        return pd.DataFrame(data)

    # Método para visualizar gráficamente las variables numéricas
    def plot(self, x):
        fig = make_subplots(rows=1, cols=2, subplot_titles=("Boxplot", "Histograma"))

        # Boxplot
        fig.add_trace(go.Box(y=self.df[x], name="Boxplot", marker_color="#008080"), row=1, col=1)

        # Histograma
        fig.add_trace(go.Histogram(x=self.df[x], name="Density", marker_color="#00a77d", nbinsx=50, histnorm='probability density'), row=1, col=2)

        fig.update_layout(showlegend=False, title_text=f"Distribución de: {x}", height=400, width=900)
        fig.show()

    # Método para graficar la matriz de correlación
    def cor_matrix(self):
        corr_matrix = self.df.corr()
        fig = go.Figure(data=go.Heatmap(
            z=corr_matrix.values,
            x=corr_matrix.columns,
            y=corr_matrix.columns,
            colorscale='Viridis',
            zmin=-1, zmax=1
        ))
        fig.update_layout(title='Matriz de Correlación', width=900, height=700)
        fig.show()

def is_na_df(df):
    na_summary = (
        df.isna().sum()
        .reset_index()
        .rename(columns={"index": "columna", 0: "n_missing"})
    )
    na_summary["total"] = len(df)
    na_summary["prop_missing"] = na_summary["n_missing"] / na_summary["total"]
    na_summary["prop_original"] = 1 - na_summary["prop_missing"]

    na_summary = na_summary[na_summary["n_missing"] > 0].reset_index(drop=True)

    total_missing = df.isna().sum().sum()
    total_elements = df.size
    resumen = pd.DataFrame({
        "columna": ["TOTAL"],
        "n_missing": [total_missing],
        "total": [total_elements],
        "prop_missing": [total_missing / total_elements],
        "prop_original": [1 - total_missing / total_elements]
    })

    return pd.concat([na_summary, resumen], ignore_index=True)

# Función auxiliar para WoE e IV
def calculate_woe_iv(df, feature, target):
    lst = []
    # Si la variable es numérica continua, la discretizamos en deciles
    if df[feature].dtype != 'O' and df[feature].nunique() > 10:
        try:
            df['bin'] = pd.qcut(df[feature], 10, duplicates='drop')
            col_name = 'bin'
        except:
            df['bin'] = df[feature] # Fallback si no puede hacer bins
            col_name = 'bin'
    else:
        df['bin'] = df[feature]
        col_name = 'bin'

    for i in df[col_name].unique():
        val = i
        lst.append({
            'Value': str(val),
            'All': df[df[col_name] == val].count()[feature],
            'Good': df[(df[col_name] == val) & (df[target] == 0)].count()[feature],
            'Bad': df[(df[col_name] == val) & (df[target] == 1)].count()[feature]
        })

    dset = pd.DataFrame(lst)
    dset['Distr_Good'] = dset['Good'] / dset['Good'].sum()
    dset['Distr_Bad'] = dset['Bad'] / dset['Bad'].sum()
    dset['WoE'] = np.log(dset['Distr_Good'] / dset['Distr_Bad'])
    dset = dset.replace({'WoE': {np.inf: 0, -np.inf: 0}}) # Tratar divisiones por cero
    dset['IV'] = (dset['Distr_Good'] - dset['Distr_Bad']) * dset['WoE']

    iv = dset['IV'].sum()
    return iv

In [21]:
df_i = pd.read_csv("../imputacion/dataset_laptops_mexico_enriquecidos.csv")

In [22]:
df_raw = pd.read_csv("../datos_crudos/dataset_laptops_mexico.csv")

In [23]:
df_raw.head()

,ID_Global,Tienda,ID_Producto,Marca,Nombre,Precio_MXN,RAM_GB,Almacenamiento_GB,Tipo_Almacenamiento,Procesador,GPU_Dedicada,Pantalla_Pulgadas,URL_Producto,Ruta_Imagen_Local
0,MX-00000,Chedraui,CHE-0000,Hp,"Laptop HP 9W149LA R5/512GB/8GB/15.6""","14,995.00",512.00,512.00,SSD,No especificado,0,15.60,https://www.chedraui.com.mx/laptop-hp-9w149la-...,NaN
1,MX-00001,Chedraui,CHE-0001,Lenovo,Laptop Lenovo 82XB001GLM Cel 8G 256 SSD 15.6 P...,"13,495.00",NaN,NaN,SSD,No especificado,0,15.60,https://www.chedraui.com.mx/laptop-lenovo-82xb...,NaN
2,MX-00002,Chedraui,CHE-0002,Acer,Laptop Acer Aspire Lite 15.6 Pulgadas FHD Core...,"17,495.00",16.00,16.00,SSD,Core I5,0,15.60,https://www.chedraui.com.mx/laptop-acer-aspire...,NaN
3,MX-00003,Chedraui,CHE-0003,Hp,"Laptop HP 255 G10 de 15,6 Pulgadas Windows 11 ...","8,495.00",8.00,8.00,SSD,Ryzen 3,0,NaN,https://www.chedraui.com.mx/laptop-hp-an0y3ata...,NaN
4,MX-00004,Chedraui,CHE-0004,Hp,Laptop HP Intel Core i5 Memoria Interna 512GB ...,"19,495.00",512.00,512.00,SSD,Core I5,0,NaN,https://www.chedraui.com.mx/laptop-hp-intel-co...,NaN


In [25]:
df_i.head()

,ID_Global,Tienda,ID_Producto,Marca,Nombre,Precio_MXN,RAM_GB,Almacenamiento_GB,Tipo_Almacenamiento,Procesador,GPU_Dedicada,Pantalla_Pulgadas,URL_Producto,Ruta_Imagen_Local,Procesador_Modelo,Numero_Nucleos,Generacion_CPU
0,MX-00000,Chedraui,CHE-0000,Hp,"Laptop HP 9W149LA R5/512GB/8GB/15.6""","14,995.00",512.00,512.00,SSD,No especificado,0,15.60,https://www.chedraui.com.mx/laptop-hp-9w149la-...,NaN,No especificado,No especificado,No especificado
1,MX-00001,Chedraui,CHE-0001,Lenovo,Laptop Lenovo 82XB001GLM Cel 8G 256 SSD 15.6 P...,"13,495.00",NaN,NaN,SSD,No especificado,0,15.60,https://www.chedraui.com.mx/laptop-lenovo-82xb...,NaN,No especificado,No especificado,No especificado
2,MX-00002,Chedraui,CHE-0002,Acer,Laptop Acer Aspire Lite 15.6 Pulgadas FHD Core...,"17,495.00",16.00,16.00,SSD,Core I5,0,15.60,https://www.chedraui.com.mx/laptop-acer-aspire...,NaN,Intel Core i5,No especificado,No especificado
3,MX-00003,Chedraui,CHE-0003,Hp,"Laptop HP 255 G10 de 15,6 Pulgadas Windows 11 ...","8,495.00",8.00,8.00,SSD,Ryzen 3,0,NaN,https://www.chedraui.com.mx/laptop-hp-an0y3ata...,NaN,AMD Ryzen 3,No especificado,No especificado
4,MX-00004,Chedraui,CHE-0004,Hp,Laptop HP Intel Core i5 Memoria Interna 512GB ...,"19,495.00",512.00,512.00,SSD,Core I5,0,NaN,https://www.chedraui.com.mx/laptop-hp-intel-co...,NaN,Intel Core i5,No especificado,No especificado


In [24]:
df_raw.shape

(35412, 14)

In [26]:
df_i.shape

(35412, 17)

* El dato de la RAM está mal y se repite con la capacidad de almacenamiento del disco.
* Variables categóricas relevantes: Marca, GPU (booleana), Procesador_Modelo, Generacion_CPU  ¿Nos conviene generar una columna para el tipo de procesador: intel, AMD, otros?
* Variables numéricas relevantes: Precio_MXN, RAM_GB, Almacenamiento_GB, Pantalla_Pulgadas, Numero_Nucleos, 

In [31]:
import pandas as pd
import numpy as np

def limpiar_ram_y_almacenamiento(df):
    # 1. Asegurar que las columnas sean numéricas para poder evaluarlas matemáticamente
    df['RAM_GB'] = pd.to_numeric(df['RAM_GB'], errors='coerce')
    df['Almacenamiento_GB'] = pd.to_numeric(df['Almacenamiento_GB'], errors='coerce')

    # 2. CASO A: Valores intercambiados (Ej: RAM = 512, Almacenamiento = 16)
    # Si la RAM es lógicamente mayor que el almacenamiento, los cruzamos.
    swapped_mask = df['RAM_GB'] > df['Almacenamiento_GB']
    df.loc[swapped_mask, ['RAM_GB', 'Almacenamiento_GB']] = df.loc[swapped_mask, ['Almacenamiento_GB', 'RAM_GB']].values

    # 3. CASO B: Valores duplicados (Ej: RAM = 16 y Almacenamiento = 16)
    dup_mask = df['RAM_GB'] == df['Almacenamiento_GB']
    
    # Si están repetidos y el valor es <= 128, es casi seguro que es la RAM.
    # Convertimos el Almacenamiento a nulo (NaN) para que tu modelo de imputación lo llene después.
    df.loc[dup_mask & (df['RAM_GB'] <= 128), 'Almacenamiento_GB'] = np.nan
    
    # Si están repetidos y el valor es > 128, es casi seguro que es el Almacenamiento.
    # Convertimos la RAM a nulo (NaN).
    df.loc[dup_mask & (df['RAM_GB'] > 128), 'RAM_GB'] = np.nan

    # 4. CASO C: Limpieza de anomalías extremas solitarias
    # Si quedó alguna RAM mayor a 128 que no estaba duplicada, la borramos por ser dato corrupto.
    df.loc[df['RAM_GB'] > 128, 'RAM_GB'] = np.nan

    return df

# Aplicar la función a tu dataset
df_laptops = limpiar_ram_y_almacenamiento(df_i)
print("Limpieza de RAM y Almacenamiento completada.")

Limpieza de RAM y Almacenamiento completada.


In [32]:
df_laptops.head()

,ID_Global,Tienda,ID_Producto,Marca,Nombre,Precio_MXN,RAM_GB,Almacenamiento_GB,Tipo_Almacenamiento,Procesador,GPU_Dedicada,Pantalla_Pulgadas,URL_Producto,Ruta_Imagen_Local,Procesador_Modelo,Numero_Nucleos,Generacion_CPU
0,MX-00000,Chedraui,CHE-0000,Hp,"Laptop HP 9W149LA R5/512GB/8GB/15.6""","14,995.00",NaN,512.00,SSD,No especificado,0,15.60,https://www.chedraui.com.mx/laptop-hp-9w149la-...,NaN,No especificado,No especificado,No especificado
1,MX-00001,Chedraui,CHE-0001,Lenovo,Laptop Lenovo 82XB001GLM Cel 8G 256 SSD 15.6 P...,"13,495.00",NaN,NaN,SSD,No especificado,0,15.60,https://www.chedraui.com.mx/laptop-lenovo-82xb...,NaN,No especificado,No especificado,No especificado
2,MX-00002,Chedraui,CHE-0002,Acer,Laptop Acer Aspire Lite 15.6 Pulgadas FHD Core...,"17,495.00",16.00,NaN,SSD,Core I5,0,15.60,https://www.chedraui.com.mx/laptop-acer-aspire...,NaN,Intel Core i5,No especificado,No especificado
3,MX-00003,Chedraui,CHE-0003,Hp,"Laptop HP 255 G10 de 15,6 Pulgadas Windows 11 ...","8,495.00",8.00,NaN,SSD,Ryzen 3,0,NaN,https://www.chedraui.com.mx/laptop-hp-an0y3ata...,NaN,AMD Ryzen 3,No especificado,No especificado
4,MX-00004,Chedraui,CHE-0004,Hp,Laptop HP Intel Core i5 Memoria Interna 512GB ...,"19,495.00",NaN,512.00,SSD,Core I5,0,NaN,https://www.chedraui.com.mx/laptop-hp-intel-co...,NaN,Intel Core i5,No especificado,No especificado


In [33]:
def asignar_marca_procesador(df, columna_origen='Procesador_Modelo'):
    # Trabajamos sobre una copia para no alterar el original inesperadamente
    df_out = df.copy()
    texto_procesadores = df_out[columna_origen].astype(str).str.lower()

    condiciones = [
        texto_procesadores.str.contains(r'intel|core i|celeron|pentium|n\d{3}', regex=True, na=False),
        texto_procesadores.str.contains(r'amd|ryzen|athlon|radeon', regex=True, na=False),
        texto_procesadores.str.contains(r'apple|m[1-4]', regex=True, na=False)
    ]

    marcas = ['Intel', 'AMD', 'Apple']

    df_out['marca_procesador'] = np.select(condiciones, marcas, default='Otros')

    mask_no_especificado = df_out[columna_origen] == 'No especificado'
    df_out.loc[mask_no_especificado, 'marca_procesador'] = 'No especificado'

    return df_out

In [34]:
df_laptops = asignar_marca_procesador(df_laptops)
print(df_laptops['marca_procesador'].value_counts())

marca_procesador
No especificado    22714
Intel               7983
AMD                 2922
Apple               1793
Name: count, dtype: int64


In [35]:
df_laptops.head()


,ID_Global,Tienda,ID_Producto,Marca,Nombre,Precio_MXN,RAM_GB,Almacenamiento_GB,Tipo_Almacenamiento,Procesador,GPU_Dedicada,Pantalla_Pulgadas,URL_Producto,Ruta_Imagen_Local,Procesador_Modelo,Numero_Nucleos,Generacion_CPU,marca_procesador
0,MX-00000,Chedraui,CHE-0000,Hp,"Laptop HP 9W149LA R5/512GB/8GB/15.6""","14,995.00",NaN,512.00,SSD,No especificado,0,15.60,https://www.chedraui.com.mx/laptop-hp-9w149la-...,NaN,No especificado,No especificado,No especificado,No especificado
1,MX-00001,Chedraui,CHE-0001,Lenovo,Laptop Lenovo 82XB001GLM Cel 8G 256 SSD 15.6 P...,"13,495.00",NaN,NaN,SSD,No especificado,0,15.60,https://www.chedraui.com.mx/laptop-lenovo-82xb...,NaN,No especificado,No especificado,No especificado,No especificado
2,MX-00002,Chedraui,CHE-0002,Acer,Laptop Acer Aspire Lite 15.6 Pulgadas FHD Core...,"17,495.00",16.00,NaN,SSD,Core I5,0,15.60,https://www.chedraui.com.mx/laptop-acer-aspire...,NaN,Intel Core i5,No especificado,No especificado,Intel
3,MX-00003,Chedraui,CHE-0003,Hp,"Laptop HP 255 G10 de 15,6 Pulgadas Windows 11 ...","8,495.00",8.00,NaN,SSD,Ryzen 3,0,NaN,https://www.chedraui.com.mx/laptop-hp-an0y3ata...,NaN,AMD Ryzen 3,No especificado,No especificado,AMD
4,MX-00004,Chedraui,CHE-0004,Hp,Laptop HP Intel Core i5 Memoria Interna 512GB ...,"19,495.00",NaN,512.00,SSD,Core I5,0,NaN,https://www.chedraui.com.mx/laptop-hp-intel-co...,NaN,Intel Core i5,No especificado,No especificado,Intel


In [41]:
palabras_clave = 'piezas|partes|reparar|refacciones'

# 3. Filtrar el DataFrame
# Usamos case=False para que no importe si están en mayúsculas o minúsculas
resultado = df_laptops[df_laptops['Nombre'].str.contains(palabras_clave, case=False, na=False)]

# 4. Mostrar los registros que coinciden
resultado[["Nombre"]]

,Nombre
3854,Hp Pavilion Dv6000 Para Refacciones (piezas) B...
3855,Piezas Para Hp Pavilion Dv6-1275la
3856,Hp Compaq Nx6320 Para Refacciones (piezas)
3857,Hp Elitebook Modelo:8560w Para Refacciones (pi...
3858,Piezas Para Laptop Hp Probook 640 Gl Piezas Pa...
...,...
35352,Laptop Retro Gateway W350a Para Piezas O Reparar
35354,Laptop Gateway Ne52203m (por Refacciones)
35356,Laptop Gateway Ms2274 Para Refacciones
35359,Laptop Gateway Nv53 Por Piezas


In [42]:
def eliminar_laptops_para_piezas(df, columna='Nombre'):
    # Patrón de palabras clave que indican equipo no funcional (regex)
    patron_basura = r'piezas|partes|reparar|refacciones|desarme|falla'
    
    # Creamos una máscara booleana ignorando mayúsculas/minúsculas
    mask_defectuosas = df[columna].str.contains(patron_basura, case=False, na=False, regex=True)
    
    # Calculamos eliminados y filtramos
    eliminadas = mask_defectuosas.sum()
    df_limpio = df[~mask_defectuosas].copy()
    
    # Mostramos el reporte exacto
    print(f"Registros eliminados (para piezas/reparar): {eliminadas}")
    print(f"Registros restantes en el dataset: {len(df_limpio)}")
    
    return df_limpio

# Aplicar la limpieza a tu dataframe actual
df_laptops = eliminar_laptops_para_piezas(df_laptops)

Registros eliminados (para piezas/reparar): 1053
Registros restantes en el dataset: 34359


In [45]:
df_laptops.head()

,ID_Global,Tienda,ID_Producto,Marca,Nombre,Precio_MXN,RAM_GB,Almacenamiento_GB,Tipo_Almacenamiento,Procesador,GPU_Dedicada,Pantalla_Pulgadas,URL_Producto,Ruta_Imagen_Local,Procesador_Modelo,Numero_Nucleos,Generacion_CPU,marca_procesador
0,MX-00000,Chedraui,CHE-0000,Hp,"Laptop HP 9W149LA R5/512GB/8GB/15.6""","14,995.00",NaN,512.00,SSD,No especificado,0,15.60,https://www.chedraui.com.mx/laptop-hp-9w149la-...,NaN,No especificado,No especificado,No especificado,No especificado
1,MX-00001,Chedraui,CHE-0001,Lenovo,Laptop Lenovo 82XB001GLM Cel 8G 256 SSD 15.6 P...,"13,495.00",NaN,NaN,SSD,No especificado,0,15.60,https://www.chedraui.com.mx/laptop-lenovo-82xb...,NaN,No especificado,No especificado,No especificado,No especificado
2,MX-00002,Chedraui,CHE-0002,Acer,Laptop Acer Aspire Lite 15.6 Pulgadas FHD Core...,"17,495.00",16.00,NaN,SSD,Core I5,0,15.60,https://www.chedraui.com.mx/laptop-acer-aspire...,NaN,Intel Core i5,No especificado,No especificado,Intel
3,MX-00003,Chedraui,CHE-0003,Hp,"Laptop HP 255 G10 de 15,6 Pulgadas Windows 11 ...","8,495.00",8.00,NaN,SSD,Ryzen 3,0,NaN,https://www.chedraui.com.mx/laptop-hp-an0y3ata...,NaN,AMD Ryzen 3,No especificado,No especificado,AMD
4,MX-00004,Chedraui,CHE-0004,Hp,Laptop HP Intel Core i5 Memoria Interna 512GB ...,"19,495.00",NaN,512.00,SSD,Core I5,0,NaN,https://www.chedraui.com.mx/laptop-hp-intel-co...,NaN,Intel Core i5,No especificado,No especificado,Intel


In [46]:
def auditar_registros_completos(df):
    columnas_clave = [
        'Precio_MXN', 
        'Marca', 
        'Almacenamiento_GB', 
        'RAM_GB', 
        'Numero_Nucleos', 
        'Procesador_Modelo', 
        'marca_procesador'
    ]
    
    # 1. Aislar solo las columnas de interés
    df_subset = df[columnas_clave]
    
    # 2. Crear máscara: que el dato no sea nulo Y que no sea "No especificado"
    mascara_completos = df_subset.notna() & (df_subset != "No especificado")
    
    # 3. Exigir que TODAS las columnas cumplan la condición en cada fila (axis=1)
    filas_validas = mascara_completos.all(axis=1)
    
    # 4. Calcular métricas
    total_completos = filas_validas.sum()
    total_registros = len(df)
    porcentaje = (total_completos / total_registros) * 100
    
    print(f"Total de registros actuales: {total_registros}")
    print(f"Registros 100% listos (Cero Nulos/No especificados): {total_completos}")
    print(f"Proporción de base limpia: {porcentaje:.2f}%")
    
    # Opcional: Extraer este dataset "de oro" en una variable nueva
    df_oro = df[filas_validas].copy()
    
    return df_oro

# Ejecutar la auditoría
df_laptops_oro = auditar_registros_completos(df_laptops)

Total de registros actuales: 34359
Registros 100% listos (Cero Nulos/No especificados): 84
Proporción de base limpia: 0.24%


In [50]:
def auditar_registros_completos(df):
    columnas_clave = [
        'Precio_MXN', 
        'Marca', 
        'RAM_GB', 
        'marca_procesador'
    ]
    
    # 1. Aislar solo las columnas de interés
    df_subset = df[columnas_clave]
    
    # 2. Crear máscara: que el dato no sea nulo Y que no sea "No especificado"
    mascara_completos = df_subset.notna() & (df_subset != "No especificado")
    
    # 3. Exigir que TODAS las columnas cumplan la condición en cada fila (axis=1)
    filas_validas = mascara_completos.all(axis=1)
    
    # 4. Calcular métricas
    total_completos = filas_validas.sum()
    total_registros = len(df)
    porcentaje = (total_completos / total_registros) * 100
    
    print(f"Total de registros actuales: {total_registros}")
    print(f"Registros 100% listos (Cero Nulos/No especificados): {total_completos}")
    print(f"Proporción de base limpia: {porcentaje:.2f}%")
    
    # Opcional: Extraer este dataset "de oro" en una variable nueva
    df_oro = df[filas_validas].copy()
    
    return df_oro

# Ejecutar la auditoría
df_laptops_oro = auditar_registros_completos(df_laptops)

Total de registros actuales: 34359
Registros 100% listos (Cero Nulos/No especificados): 10683
Proporción de base limpia: 31.09%


In [51]:
# Ver el porcentaje de datos válidos por columna individual
cols_revision = ['Precio_MXN', 'Marca', 'RAM_GB', 'marca_procesador', 'Almacenamiento_GB', 'Procesador_Modelo', 'Numero_Nucleos']

for col in cols_revision:
    validos = (df_laptops[col].notna()) & (df_laptops[col] != "No especificado")
    pct = (validos.sum() / len(df_laptops)) * 100
    print(f"Columna '{col}': {validos.sum()} registros válidos ({pct:.2f}%)")

Columna 'Precio_MXN': 34359 registros válidos (100.00%)
Columna 'Marca': 34359 registros válidos (100.00%)
Columna 'RAM_GB': 26225 registros válidos (76.33%)
Columna 'marca_procesador': 12693 registros válidos (36.94%)
Columna 'Almacenamiento_GB': 3509 registros válidos (10.21%)
Columna 'Procesador_Modelo': 12693 registros válidos (36.94%)
Columna 'Numero_Nucleos': 4934 registros válidos (14.36%)


In [53]:
df_laptops[['Nombre', 'Almacenamiento_GB']].head(20)

,Nombre,Almacenamiento_GB
0,"Laptop HP 9W149LA R5/512GB/8GB/15.6""",512.00
1,Laptop Lenovo 82XB001GLM Cel 8G 256 SSD 15.6 P...,NaN
2,Laptop Acer Aspire Lite 15.6 Pulgadas FHD Core...,NaN
3,"Laptop HP 255 G10 de 15,6 Pulgadas Windows 11 ...",NaN
4,Laptop HP Intel Core i5 Memoria Interna 512GB ...,512.00
5,Laptop Lenovo S3 RAM 8G 512SSD 15.6 Pulgadas A...,NaN
6,"Laptop Gaming Lenovo LOQ 15ARP9 15.6"" FHD AMD ...",NaN
7,"Laptop Lenovo Ideapad Slim 3 15IAN8 15.6"" FHD ...",NaN
8,Laptop Lanix 14 Pulgadas 8GB RAM 256GB de Alma...,NaN
9,Laptop Asus VivoBook Go 14 Pulgadas FHD Intel ...,NaN


In [56]:
def rescatar_almacenamiento_desde_nombre(df, col_nombre='Nombre', col_alm='Almacenamiento_GB', col_ram='RAM_GB'):
    df_out = df.copy()
    
    # Trabajamos únicamente sobre los registros donde Almacenamiento es nulo
    mask_nan = df_out[col_alm].isna()
    
    def extraer_gb(row):
        texto = str(row[col_nombre]).upper()
        ram_val = row[col_ram] if col_ram in row and pd.notna(row[col_ram]) else None
        
        # 1. Caso Terabytes (ej: 1TB, 2TB) -> Convertir a GB (1024, 2048)
        match_tb = re.search(r'(\d+)\s*TB', texto)
        if match_tb:
            return int(match_tb.group(1)) * 1024
            
        # 2. Caso Gigabytes (ej: 128GB, 256GB, 512GB)
        # Buscamos todas las apariciones de números seguidos de GB
        matches_gb = re.findall(r'(\d+)\s*GB', texto)
        if matches_gb:
            numeros = [int(m) for m in matches_gb]
            
            # El almacenamiento en laptops comercialmente suele ser >= 128 GB
            almacenamientos = [n for n in numeros if n >= 128]
            if almacenamientos:
                return almacenamientos[0]
            
            # Si hay un número diferente a la RAM conocida y es mayor que la RAM, tomarlo
            if ram_val:
                diferentes_a_ram = [n for n in numeros if n != int(ram_val) and n > ram_val]
                if diferentes_a_ram:
                    return diferentes_a_ram[0]
                    
        # 3. Caso Mención explícita de disco (ej: 512 SSD, 256 SSD)
        match_disco = re.search(r'(\d+)\s*(?:GB)?\s*(?:SSD|HDD|NVME)', texto)
        if match_disco:
            val = int(match_disco.group(1))
            if val >= 64: # Acepta desde eMMC/SSD básicos de 64GB
                return val
                
        return np.nan

    # Aplicamos la función solo a las filas con NaN en Almacenamiento
    almacenamiento_recuperado = df_out[mask_nan].apply(extraer_gb, axis=1)
    
    # Actualizamos el DataFrame
    df_out.loc[mask_nan, col_alm] = almacenamiento_recuperado
    
    # Reporte de resultados
    total_recuperados = df_out[col_alm].notna().sum() - (~mask_nan).sum()
    print(f"Registros de Almacenamiento recuperados desde 'Nombre': {total_recuperados}")
    print(f"Nulos restantes en '{col_alm}': {df_out[col_alm].isna().sum()}")
    
    return df_out

# Ejecutar el rescate sobre tu DataFrame
df_laptops = rescatar_almacenamiento_desde_nombre(df_laptops)

Registros de Almacenamiento recuperados desde 'Nombre': 24495
Nulos restantes en 'Almacenamiento_GB': 6355


In [57]:
# Ver el porcentaje de datos válidos por columna individual
cols_revision = ['Precio_MXN', 'Marca', 'RAM_GB', 'marca_procesador', 'Almacenamiento_GB', 'Procesador_Modelo', 'Numero_Nucleos']

for col in cols_revision:
    validos = (df_laptops[col].notna()) & (df_laptops[col] != "No especificado")
    pct = (validos.sum() / len(df_laptops)) * 100
    print(f"Columna '{col}': {validos.sum()} registros válidos ({pct:.2f}%)")

Columna 'Precio_MXN': 34359 registros válidos (100.00%)
Columna 'Marca': 34359 registros válidos (100.00%)
Columna 'RAM_GB': 26225 registros válidos (76.33%)
Columna 'marca_procesador': 12693 registros válidos (36.94%)
Columna 'Almacenamiento_GB': 28004 registros válidos (81.50%)
Columna 'Procesador_Modelo': 12693 registros válidos (36.94%)
Columna 'Numero_Nucleos': 4934 registros válidos (14.36%)


In [59]:
pd.set_option('display.max_colwidth', None)

In [69]:
df_laptops[['Nombre']].iloc[95:98]

,Nombre
95,Escritorio Multifuncional Movil Aquila Blanco con Beige EM4596
96,Mesa Auxiliar FurnitureR Haya Bello
97,Mesa Auxiliar FurnitureR Negro Bello


In [71]:
import pandas as pd

def eliminar_productos_no_laptop(df, columna='Nombre'):
    # Lista de palabras clave asociadas a muebles, accesorios y refacciones sueltas
    patrones_no_laptop = [
        r'escritorio', r'mesa', r'silla', r'mueble',
        r'funda', r'mochila', r'estuche', r'portafolio', r'morral',
        r'cargador', r'bater[ií]a', r'eliminador', r'cable', r'adaptador',
        r'soporte', r'base\s+enfriadora', r'base\s+refrigerante',
        r'vinilo', r'skin', r'sticker', r'mica', r'protector',
        r'teclado', r'mouse', r'carcasa', r'bisagra', r'pantalla\s+de\s+repuesto'
    ]
    
    # Unimos todos los patrones con un OR en regex (\b asegura límites de palabra)
    patron_regex = r'\b(?:' + '|'.join(patrones_no_laptop) + r')\b'    
    # Identificamos las filas que coinciden con algún producto ajeno
    mask_ajenos = df[columna].str.contains(patron_regex, case=False, na=False, regex=True)
    
    # Conteo y filtrado
    eliminados = mask_ajenos.sum()
    df_limpio = df[~mask_ajenos].copy()
    
    print(f"Productos ajenos (muebles/accesorios) eliminados: {eliminados}")
    print(f"Registros restantes en el dataset: {len(df_limpio)}")
    
    return df_limpio

# Ejecutar la limpieza en tu DataFrame
df_laptops = eliminar_productos_no_laptop(df_laptops)

Productos ajenos (muebles/accesorios) eliminados: 3002
Registros restantes en el dataset: 31357


/tmp/ipykernel_4117/151563449.py:18: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  mask_ajenos = df[columna].str.contains(patron_regex, case=False, na=False, regex=True)


In [76]:
df_laptops[['Procesador_Modelo', 'Numero_Nucleos',"Generacion_CPU"]].head(20)

,Procesador_Modelo,Numero_Nucleos,Generacion_CPU
0,No especificado,No especificado,No especificado
1,No especificado,No especificado,No especificado
2,Intel Core i5,No especificado,No especificado
3,AMD Ryzen 3,No especificado,No especificado
4,Intel Core i5,No especificado,No especificado
5,No especificado,No especificado,No especificado
6,AMD Ryzen 5,No especificado,No especificado
7,Intel Core i3,No especificado,No especificado
8,No especificado,No especificado,No especificado
9,Intel Core i3-N305,8,Alder Lake-N
